# Significance test

For all model comparisons, statistical significance was assessed using paired tests on per-user metric values.  
Because multiple models were compared, p-values were corrected using the Holm–Bonferroni procedure.  
The same correction protocol was applied consistently across all XGBoost and simulation experiments.

### Logirstic Regression baseline multiseed vs network mulitseed

In [12]:
import json
import numpy as np
from scipy import stats
from pathlib import Path

base = Path(r"C:\Users\cieci\OneDrive\Dokumenty\GitHub\bachelor.project\models\2_Logistic_Regression_model\results")

with open(base / "results_lr_baseline_multiseed.json") as f:
    lr_no = json.load(f)

with open(base / "results_lr_network_multiseed.json") as f:
    lr_with = json.load(f)

METRICS = [
    "HitRate@1", "HitRate@5", "HitRate@10", "HitRate@20",
    "NDCG@1",    "NDCG@5",    "NDCG@10",    "NDCG@20",
    "MRR",
]

ALPHA = 0.05


def extract(per_seed_list, metric):
    return np.array([s[metric] for s in per_seed_list], dtype=float)


def holm_correction(p_values, alpha=0.05):
    """
    Holm-Bonferroni correction.
    Returns adjusted p-values and significance decisions.
    """
    p_values = np.asarray(p_values, dtype=float)
    m = len(p_values)

    order = np.argsort(p_values)
    sorted_p = p_values[order]

    adjusted_sorted = np.empty(m)

    for i, p in enumerate(sorted_p):
        adjusted_sorted[i] = (m - i) * p

    adjusted_sorted = np.maximum.accumulate(adjusted_sorted)
    adjusted_sorted = np.minimum(adjusted_sorted, 1.0)

    adjusted = np.empty(m)
    adjusted[order] = adjusted_sorted

    significant = adjusted < alpha
    return adjusted, significant


def run_tests(no_data, with_data, split="val"):
    key = f"per_seed_{split}"
    results = []

    for metric in METRICS:
        a = extract(no_data[key], metric)
        b = extract(with_data[key], metric)
        diff = b - a

        t_stat, p_val = stats.ttest_rel(b, a)

        results.append({
            "metric": metric,
            "mean_no": a.mean(),
            "mean_with": b.mean(),
            "diff": diff.mean(),
            "diff_std": diff.std(ddof=1),
            "t": t_stat,
            "p_raw": p_val,
        })

    raw_p_values = [r["p_raw"] for r in results]
    p_holm, sig_holm = holm_correction(raw_p_values, alpha=ALPHA)

    for r, p_adj, sig in zip(results, p_holm, sig_holm):
        r["p_holm"] = p_adj
        r["sig"] = sig

    return results


def print_table(results, title):
    print(f"\n{'═'*98}")
    print(f"  {title}")
    print(f"{'═'*98}")
    header = (
        f"{'Metric':<14} {'no_net':>8} {'with_net':>9} "
        f"{'Δ (with−no)':>12} {'t':>7} {'p_raw':>10} {'p_holm':>10}  sig?"
    )
    print(header)
    print("─" * 98)

    for r in results:
        flag = "✓ YES" if r["sig"] else "  no"
        delta_sign = "+" if r["diff"] >= 0 else ""

        print(
            f"{r['metric']:<14}"
            f" {r['mean_no']:>8.4f}"
            f" {r['mean_with']:>9.4f}"
            f" {delta_sign}{r['diff']:>+10.4f}"
            f" {r['t']:>7.3f}"
            f" {r['p_raw']:>10.4f}"
            f" {r['p_holm']:>10.4f}"
            f"  {flag}"
        )

    print("─" * 98)
    n_sig = sum(r["sig"] for r in results)
    print(f"  Significant after Holm correction at α={ALPHA}: {n_sig}/{len(results)} metrics")


def cohens_d(no_data, with_data, split, metric):
    key = f"per_seed_{split}"
    a = extract(no_data[key], metric)
    b = extract(with_data[key], metric)
    diff = b - a

    std = diff.std(ddof=1)
    if std == 0:
        return np.inf if diff.mean() != 0 else 0.0

    return diff.mean() / std


def interpret(d):
    ad = abs(d)
    if ad < 0.2:
        return "negligible"
    if ad < 0.5:
        return "small"
    if ad < 0.8:
        return "medium"
    return "large"


all_results = {}

for split in ("val", "test"):
    res = run_tests(lr_no, lr_with, split)
    all_results[split] = res
    print_table(res, f"Logistic Regression — {split.upper()}")

print("\n\nEffect sizes (Cohen's d) for Holm-significant results:")
print(f"{'Model':<14} {'Split':<6} {'Metric':<14} {'Cohen d':>9}  interpretation")
print("─" * 65)

for split in ("val", "test"):
    for r in all_results[split]:
        if r["sig"]:
            d = cohens_d(lr_no, lr_with, split, r["metric"])
            print(f"{'LR':<14} {split:<6} {r['metric']:<14} {d:>9.3f}  {interpret(d)}")


══════════════════════════════════════════════════════════════════════════════════════════════════
  Logistic Regression — VAL
══════════════════════════════════════════════════════════════════════════════════════════════════
Metric           no_net  with_net  Δ (with−no)       t      p_raw     p_holm  sig?
──────────────────────────────────────────────────────────────────────────────────────────────────
HitRate@1        0.1616    0.7749 +   +0.6134 1068.797     0.0000     0.0000  ✓ YES
HitRate@5        0.5689    0.9168 +   +0.3479 766.137     0.0000     0.0000  ✓ YES
HitRate@10       0.7480    0.9452 +   +0.1972 312.887     0.0000     0.0000  ✓ YES
HitRate@20       0.8434    0.9679 +   +0.1244 425.383     0.0000     0.0000  ✓ YES
NDCG@1           0.1616    0.7749 +   +0.6134 1068.797     0.0000     0.0000  ✓ YES
NDCG@5           0.3669    0.8554 +   +0.4886 2772.879     0.0000     0.0000  ✓ YES
NDCG@10          0.4257    0.8647 +   +0.4390 3032.001     0.0000     0.0000  ✓ YES
NDCG@2

### Random Forest baseline multiseed vs network mulitseed

In [11]:
import json
import numpy as np
from scipy import stats
from pathlib import Path

base = Path(r"C:\Users\cieci\OneDrive\Dokumenty\GitHub\bachelor.project\models\3_Random_Forest_model\results")

with open(base / "results_rf_baseline_multiseed.json") as f:
    lr_no = json.load(f)

with open(base / "results_rf_network_multiseed.json") as f:
    lr_with = json.load(f)

METRICS = [
    "HitRate@1", "HitRate@5", "HitRate@10", "HitRate@20",
    "NDCG@1",    "NDCG@5",    "NDCG@10",    "NDCG@20",
    "MRR",
]

ALPHA = 0.05


def extract(per_seed_list, metric):
    return np.array([s[metric] for s in per_seed_list], dtype=float)


def holm_correction(p_values, alpha=0.05):
    """
    Holm-Bonferroni correction.
    Returns adjusted p-values and significance decisions.
    """
    p_values = np.asarray(p_values, dtype=float)
    m = len(p_values)

    order = np.argsort(p_values)
    sorted_p = p_values[order]

    adjusted_sorted = np.empty(m)

    for i, p in enumerate(sorted_p):
        adjusted_sorted[i] = (m - i) * p

    adjusted_sorted = np.maximum.accumulate(adjusted_sorted)
    adjusted_sorted = np.minimum(adjusted_sorted, 1.0)

    adjusted = np.empty(m)
    adjusted[order] = adjusted_sorted

    significant = adjusted < alpha
    return adjusted, significant


def run_tests(no_data, with_data, split="val"):
    key = f"per_seed_{split}"
    results = []

    for metric in METRICS:
        a = extract(no_data[key], metric)
        b = extract(with_data[key], metric)
        diff = b - a

        t_stat, p_val = stats.ttest_rel(b, a)

        results.append({
            "metric": metric,
            "mean_no": a.mean(),
            "mean_with": b.mean(),
            "diff": diff.mean(),
            "diff_std": diff.std(ddof=1),
            "t": t_stat,
            "p_raw": p_val,
        })

    raw_p_values = [r["p_raw"] for r in results]
    p_holm, sig_holm = holm_correction(raw_p_values, alpha=ALPHA)

    for r, p_adj, sig in zip(results, p_holm, sig_holm):
        r["p_holm"] = p_adj
        r["sig"] = sig

    return results


def print_table(results, title):
    print(f"\n{'═'*98}")
    print(f"  {title}")
    print(f"{'═'*98}")
    header = (
        f"{'Metric':<14} {'no_net':>8} {'with_net':>9} "
        f"{'Δ (with−no)':>12} {'t':>7} {'p_raw':>10} {'p_holm':>10}  sig?"
    )
    print(header)
    print("─" * 98)

    for r in results:
        flag = "✓ YES" if r["sig"] else "  no"
        delta_sign = "+" if r["diff"] >= 0 else ""

        print(
            f"{r['metric']:<14}"
            f" {r['mean_no']:>8.4f}"
            f" {r['mean_with']:>9.4f}"
            f" {delta_sign}{r['diff']:>+10.4f}"
            f" {r['t']:>7.3f}"
            f" {r['p_raw']:>10.4f}"
            f" {r['p_holm']:>10.4f}"
            f"  {flag}"
        )

    print("─" * 98)
    n_sig = sum(r["sig"] for r in results)
    print(f"  Significant after Holm correction at α={ALPHA}: {n_sig}/{len(results)} metrics")


def cohens_d(no_data, with_data, split, metric):
    key = f"per_seed_{split}"
    a = extract(no_data[key], metric)
    b = extract(with_data[key], metric)
    diff = b - a

    std = diff.std(ddof=1)
    if std == 0:
        return np.inf if diff.mean() != 0 else 0.0

    return diff.mean() / std


def interpret(d):
    ad = abs(d)
    if ad < 0.2:
        return "negligible"
    if ad < 0.5:
        return "small"
    if ad < 0.8:
        return "medium"
    return "large"


all_results = {}

for split in ("val", "test"):
    res = run_tests(lr_no, lr_with, split)
    all_results[split] = res
    print_table(res, f"Random Forest — {split.upper()}")

print("\n\nEffect sizes (Cohen's d) for Holm-significant results:")
print(f"{'Model':<14} {'Split':<6} {'Metric':<14} {'Cohen d':>9}  interpretation")
print("─" * 65)

for split in ("val", "test"):
    for r in all_results[split]:
        if r["sig"]:
            d = cohens_d(lr_no, lr_with, split, r["metric"])
            print(f"{'LR':<14} {split:<6} {r['metric']:<14} {d:>9.3f}  {interpret(d)}")


══════════════════════════════════════════════════════════════════════════════════════════════════
  Random Forest — VAL
══════════════════════════════════════════════════════════════════════════════════════════════════
Metric           no_net  with_net  Δ (with−no)       t      p_raw     p_holm  sig?
──────────────────────────────────────────────────────────────────────────────────────────────────
HitRate@1        0.6204    0.9587 +   +0.3383  92.325     0.0001     0.0003  ✓ YES
HitRate@5        0.7775    0.9723 +   +0.1948 168.439     0.0000     0.0003  ✓ YES
HitRate@10       0.8347    0.9788 +   +0.1442 125.295     0.0001     0.0003  ✓ YES
HitRate@20       0.8991    0.9869 +   +0.0878 214.187     0.0000     0.0002  ✓ YES
NDCG@1           0.6204    0.9587 +   +0.3383  92.325     0.0001     0.0003  ✓ YES
NDCG@5           0.7056    0.9659 +   +0.2604 154.264     0.0000     0.0003  ✓ YES
NDCG@10          0.7240    0.9681 +   +0.2441 155.701     0.0000     0.0003  ✓ YES
NDCG@20         

### XGBoost baseline multiseed vs network mulitseed

In [10]:
import json
import numpy as np
from scipy import stats
from pathlib import Path

base = Path(r"C:\Users\cieci\OneDrive\Dokumenty\GitHub\bachelor.project\models\4_XGBoost_model\results")

with open(base / "results_xgb_baseline_multiseed.json") as f:
    lr_no = json.load(f)

with open(base / "results_xgb_network_multiseed.json") as f:
    lr_with = json.load(f)

METRICS = [
    "HitRate@1", "HitRate@5", "HitRate@10", "HitRate@20",
    "NDCG@1",    "NDCG@5",    "NDCG@10",    "NDCG@20",
    "MRR",
]

ALPHA = 0.05


def extract(per_seed_list, metric):
    return np.array([s[metric] for s in per_seed_list], dtype=float)


def holm_correction(p_values, alpha=0.05):
    """
    Holm-Bonferroni correction.
    Returns adjusted p-values and significance decisions.
    """
    p_values = np.asarray(p_values, dtype=float)
    m = len(p_values)

    order = np.argsort(p_values)
    sorted_p = p_values[order]

    adjusted_sorted = np.empty(m)

    for i, p in enumerate(sorted_p):
        adjusted_sorted[i] = (m - i) * p

    adjusted_sorted = np.maximum.accumulate(adjusted_sorted)
    adjusted_sorted = np.minimum(adjusted_sorted, 1.0)

    adjusted = np.empty(m)
    adjusted[order] = adjusted_sorted

    significant = adjusted < alpha
    return adjusted, significant


def run_tests(no_data, with_data, split="val"):
    key = f"per_seed_{split}"
    results = []

    for metric in METRICS:
        a = extract(no_data[key], metric)
        b = extract(with_data[key], metric)
        diff = b - a

        t_stat, p_val = stats.ttest_rel(b, a)

        results.append({
            "metric": metric,
            "mean_no": a.mean(),
            "mean_with": b.mean(),
            "diff": diff.mean(),
            "diff_std": diff.std(ddof=1),
            "t": t_stat,
            "p_raw": p_val,
        })

    raw_p_values = [r["p_raw"] for r in results]
    p_holm, sig_holm = holm_correction(raw_p_values, alpha=ALPHA)

    for r, p_adj, sig in zip(results, p_holm, sig_holm):
        r["p_holm"] = p_adj
        r["sig"] = sig

    return results


def print_table(results, title):
    print(f"\n{'═'*98}")
    print(f"  {title}")
    print(f"{'═'*98}")
    header = (
        f"{'Metric':<14} {'no_net':>8} {'with_net':>9} "
        f"{'Δ (with−no)':>12} {'t':>7} {'p_raw':>10} {'p_holm':>10}  sig?"
    )
    print(header)
    print("─" * 98)

    for r in results:
        flag = "✓ YES" if r["sig"] else "  no"
        delta_sign = "+" if r["diff"] >= 0 else ""

        print(
            f"{r['metric']:<14}"
            f" {r['mean_no']:>8.4f}"
            f" {r['mean_with']:>9.4f}"
            f" {delta_sign}{r['diff']:>+10.4f}"
            f" {r['t']:>7.3f}"
            f" {r['p_raw']:>10.4f}"
            f" {r['p_holm']:>10.4f}"
            f"  {flag}"
        )

    print("─" * 98)
    n_sig = sum(r["sig"] for r in results)
    print(f"  Significant after Holm correction at α={ALPHA}: {n_sig}/{len(results)} metrics")


def cohens_d(no_data, with_data, split, metric):
    key = f"per_seed_{split}"
    a = extract(no_data[key], metric)
    b = extract(with_data[key], metric)
    diff = b - a

    std = diff.std(ddof=1)
    if std == 0:
        return np.inf if diff.mean() != 0 else 0.0

    return diff.mean() / std


def interpret(d):
    ad = abs(d)
    if ad < 0.2:
        return "negligible"
    if ad < 0.5:
        return "small"
    if ad < 0.8:
        return "medium"
    return "large"


all_results = {}

for split in ("val", "test"):
    res = run_tests(lr_no, lr_with, split)
    all_results[split] = res
    print_table(res, f"XGBoost — {split.upper()}")

print("\n\nEffect sizes (Cohen's d) for Holm-significant results:")
print(f"{'Model':<14} {'Split':<6} {'Metric':<14} {'Cohen d':>9}  interpretation")
print("─" * 65)

for split in ("val", "test"):
    for r in all_results[split]:
        if r["sig"]:
            d = cohens_d(lr_no, lr_with, split, r["metric"])
            print(f"{'LR':<14} {split:<6} {r['metric']:<14} {d:>9.3f}  {interpret(d)}")


══════════════════════════════════════════════════════════════════════════════════════════════════
  XGBoost — VAL
══════════════════════════════════════════════════════════════════════════════════════════════════
Metric           no_net  with_net  Δ (with−no)       t      p_raw     p_holm  sig?
──────────────────────────────────────────────────────────────────────────────────────────────────
HitRate@1        0.7744    0.9949 +   +0.2205 384.457     0.0000     0.0000  ✓ YES
HitRate@5        0.8625    0.9970 +   +0.1345  80.937     0.0002     0.0005  ✓ YES
HitRate@10       0.9002    0.9977 +   +0.0974  60.909     0.0003     0.0005  ✓ YES
HitRate@20       0.9399    0.9984 +   +0.0585  57.018     0.0003     0.0005  ✓ YES
NDCG@1           0.7744    0.9949 +   +0.2205 384.457     0.0000     0.0000  ✓ YES
NDCG@5           0.8214    0.9960 +   +0.1746 267.830     0.0000     0.0001  ✓ YES
NDCG@10          0.8336    0.9962 +   +0.1627 452.135     0.0000     0.0000  ✓ YES
NDCG@20          0.843

### Neural Network baseline multiseed vs network mulitseed

In [9]:
import json
import numpy as np
from scipy import stats
from pathlib import Path

base = Path(r"C:\Users\cieci\OneDrive\Dokumenty\GitHub\bachelor.project\models\5_neural_Network_model\results")

with open(base / "results_NN_baseline_multiseed.json") as f:
    lr_no = json.load(f)

with open(base / "results_NN_network_multiseed.json") as f:
    lr_with = json.load(f)

METRICS = [
    "HitRate@1", "HitRate@5", "HitRate@10", "HitRate@20",
    "NDCG@1",    "NDCG@5",    "NDCG@10",    "NDCG@20",
    "MRR",
]

ALPHA = 0.05


def extract(per_seed_list, metric):
    return np.array([s[metric] for s in per_seed_list], dtype=float)


def holm_correction(p_values, alpha=0.05):
    """
    Holm-Bonferroni correction.
    Returns adjusted p-values and significance decisions.
    """
    p_values = np.asarray(p_values, dtype=float)
    m = len(p_values)

    order = np.argsort(p_values)
    sorted_p = p_values[order]

    adjusted_sorted = np.empty(m)

    for i, p in enumerate(sorted_p):
        adjusted_sorted[i] = (m - i) * p

    adjusted_sorted = np.maximum.accumulate(adjusted_sorted)
    adjusted_sorted = np.minimum(adjusted_sorted, 1.0)

    adjusted = np.empty(m)
    adjusted[order] = adjusted_sorted

    significant = adjusted < alpha
    return adjusted, significant


def run_tests(no_data, with_data, split="val"):
    key = f"per_seed_{split}"
    results = []

    for metric in METRICS:
        a = extract(no_data[key], metric)
        b = extract(with_data[key], metric)
        diff = b - a

        t_stat, p_val = stats.ttest_rel(b, a)

        results.append({
            "metric": metric,
            "mean_no": a.mean(),
            "mean_with": b.mean(),
            "diff": diff.mean(),
            "diff_std": diff.std(ddof=1),
            "t": t_stat,
            "p_raw": p_val,
        })

    raw_p_values = [r["p_raw"] for r in results]
    p_holm, sig_holm = holm_correction(raw_p_values, alpha=ALPHA)

    for r, p_adj, sig in zip(results, p_holm, sig_holm):
        r["p_holm"] = p_adj
        r["sig"] = sig

    return results


def print_table(results, title):
    print(f"\n{'═'*98}")
    print(f"  {title}")
    print(f"{'═'*98}")
    header = (
        f"{'Metric':<14} {'no_net':>8} {'with_net':>9} "
        f"{'Δ (with−no)':>12} {'t':>7} {'p_raw':>10} {'p_holm':>10}  sig?"
    )
    print(header)
    print("─" * 98)

    for r in results:
        flag = "✓ YES" if r["sig"] else "  no"
        delta_sign = "+" if r["diff"] >= 0 else ""

        print(
            f"{r['metric']:<14}"
            f" {r['mean_no']:>8.4f}"
            f" {r['mean_with']:>9.4f}"
            f" {delta_sign}{r['diff']:>+10.4f}"
            f" {r['t']:>7.3f}"
            f" {r['p_raw']:>10.4f}"
            f" {r['p_holm']:>10.4f}"
            f"  {flag}"
        )

    print("─" * 98)
    n_sig = sum(r["sig"] for r in results)
    print(f"  Significant after Holm correction at α={ALPHA}: {n_sig}/{len(results)} metrics")


def cohens_d(no_data, with_data, split, metric):
    key = f"per_seed_{split}"
    a = extract(no_data[key], metric)
    b = extract(with_data[key], metric)
    diff = b - a

    std = diff.std(ddof=1)
    if std == 0:
        return np.inf if diff.mean() != 0 else 0.0

    return diff.mean() / std


def interpret(d):
    ad = abs(d)
    if ad < 0.2:
        return "negligible"
    if ad < 0.5:
        return "small"
    if ad < 0.8:
        return "medium"
    return "large"


all_results = {}

for split in ("val", "test"):
    res = run_tests(lr_no, lr_with, split)
    all_results[split] = res
    print_table(res, f"Neural Network — {split.upper()}")

print("\n\nEffect sizes (Cohen's d) for Holm-significant results:")
print(f"{'Model':<14} {'Split':<6} {'Metric':<14} {'Cohen d':>9}  interpretation")
print("─" * 65)

for split in ("val", "test"):
    for r in all_results[split]:
        if r["sig"]:
            d = cohens_d(lr_no, lr_with, split, r["metric"])
            print(f"{'LR':<14} {split:<6} {r['metric']:<14} {d:>9.3f}  {interpret(d)}")


══════════════════════════════════════════════════════════════════════════════════════════════════
  Neural Network — VAL
══════════════════════════════════════════════════════════════════════════════════════════════════
Metric           no_net  with_net  Δ (with−no)       t      p_raw     p_holm  sig?
──────────────────────────────────────────────────────────────────────────────────────────────────
HitRate@1        0.1848    0.1826    -0.0022  -1.932     0.1931     1.0000    no
HitRate@5        0.4871    0.4904 +   +0.0033   0.690     0.5616     1.0000    no
HitRate@10       0.6859    0.6886 +   +0.0027   0.557     0.6337     1.0000    no
HitRate@20       0.8767    0.8801 +   +0.0034   1.745     0.2231     1.0000    no
NDCG@1           0.1848    0.1826    -0.0022  -1.932     0.1931     1.0000    no
NDCG@5           0.3382    0.3385 +   +0.0004   0.149     0.8949     1.0000    no
NDCG@10          0.4023    0.4026 +   +0.0002   0.091     0.9358     1.0000    no
NDCG@20          0.4508 

### GraphSAGE baseline multiseed vs network mulitseed

In [13]:
import json
import numpy as np
from scipy import stats
from pathlib import Path

base = Path(r"C:\Users\cieci\OneDrive\Dokumenty\GitHub\bachelor.project\models\6_GraphSAGE_model\results")

with open(base / "results_GraphSAGE_baseline_multiseed.json") as f:
    lr_no = json.load(f)

with open(base / "results_GraphSAGE_network_multiseed.json") as f:
    lr_with = json.load(f)

METRICS = [
    "HitRate@1", "HitRate@5", "HitRate@10", "HitRate@20",
    "NDCG@1",    "NDCG@5",    "NDCG@10",    "NDCG@20",
    "MRR",
]

ALPHA = 0.05


def extract(per_seed_list, metric):
    return np.array([s[metric] for s in per_seed_list], dtype=float)


def holm_correction(p_values, alpha=0.05):
    """
    Holm-Bonferroni correction.
    Returns adjusted p-values and significance decisions.
    """
    p_values = np.asarray(p_values, dtype=float)
    m = len(p_values)

    order = np.argsort(p_values)
    sorted_p = p_values[order]

    adjusted_sorted = np.empty(m)

    for i, p in enumerate(sorted_p):
        adjusted_sorted[i] = (m - i) * p

    adjusted_sorted = np.maximum.accumulate(adjusted_sorted)
    adjusted_sorted = np.minimum(adjusted_sorted, 1.0)

    adjusted = np.empty(m)
    adjusted[order] = adjusted_sorted

    significant = adjusted < alpha
    return adjusted, significant


def run_tests(no_data, with_data, split="val"):
    key = f"per_seed_{split}"
    results = []

    for metric in METRICS:
        a = extract(no_data[key], metric)
        b = extract(with_data[key], metric)
        diff = b - a

        t_stat, p_val = stats.ttest_rel(b, a)

        results.append({
            "metric": metric,
            "mean_no": a.mean(),
            "mean_with": b.mean(),
            "diff": diff.mean(),
            "diff_std": diff.std(ddof=1),
            "t": t_stat,
            "p_raw": p_val,
        })

    raw_p_values = [r["p_raw"] for r in results]
    p_holm, sig_holm = holm_correction(raw_p_values, alpha=ALPHA)

    for r, p_adj, sig in zip(results, p_holm, sig_holm):
        r["p_holm"] = p_adj
        r["sig"] = sig

    return results


def print_table(results, title):
    print(f"\n{'═'*98}")
    print(f"  {title}")
    print(f"{'═'*98}")
    header = (
        f"{'Metric':<14} {'no_net':>8} {'with_net':>9} "
        f"{'Δ (with−no)':>12} {'t':>7} {'p_raw':>10} {'p_holm':>10}  sig?"
    )
    print(header)
    print("─" * 98)

    for r in results:
        flag = "✓ YES" if r["sig"] else "  no"
        delta_sign = "+" if r["diff"] >= 0 else ""

        print(
            f"{r['metric']:<14}"
            f" {r['mean_no']:>8.4f}"
            f" {r['mean_with']:>9.4f}"
            f" {delta_sign}{r['diff']:>+10.4f}"
            f" {r['t']:>7.3f}"
            f" {r['p_raw']:>10.4f}"
            f" {r['p_holm']:>10.4f}"
            f"  {flag}"
        )

    print("─" * 98)
    n_sig = sum(r["sig"] for r in results)
    print(f"  Significant after Holm correction at α={ALPHA}: {n_sig}/{len(results)} metrics")


def cohens_d(no_data, with_data, split, metric):
    key = f"per_seed_{split}"
    a = extract(no_data[key], metric)
    b = extract(with_data[key], metric)
    diff = b - a

    std = diff.std(ddof=1)
    if std == 0:
        return np.inf if diff.mean() != 0 else 0.0

    return diff.mean() / std


def interpret(d):
    ad = abs(d)
    if ad < 0.2:
        return "negligible"
    if ad < 0.5:
        return "small"
    if ad < 0.8:
        return "medium"
    return "large"


all_results = {}

for split in ("val", "test"):
    res = run_tests(lr_no, lr_with, split)
    all_results[split] = res
    print_table(res, f"GraphSAGE — {split.upper()}")

print("\n\nEffect sizes (Cohen's d) for Holm-significant results:")
print(f"{'Model':<14} {'Split':<6} {'Metric':<14} {'Cohen d':>9}  interpretation")
print("─" * 65)

for split in ("val", "test"):
    for r in all_results[split]:
        if r["sig"]:
            d = cohens_d(lr_no, lr_with, split, r["metric"])
            print(f"{'LR':<14} {split:<6} {r['metric']:<14} {d:>9.3f}  {interpret(d)}")


══════════════════════════════════════════════════════════════════════════════════════════════════
  GraphSAGE — VAL
══════════════════════════════════════════════════════════════════════════════════════════════════
Metric           no_net  with_net  Δ (with−no)       t      p_raw     p_holm  sig?
──────────────────────────────────────────────────────────────────────────────────────────────────
HitRate@1        0.1971    0.2017 +   +0.0046   1.248     0.3383     1.0000    no
HitRate@5        0.5366    0.5424 +   +0.0058   2.323     0.1458     1.0000    no
HitRate@10       0.7329    0.7375 +   +0.0045   2.377     0.1406     1.0000    no
HitRate@20       0.9026    0.9049 +   +0.0023   1.853     0.2051     1.0000    no
NDCG@1           0.1971    0.2017 +   +0.0046   1.248     0.3383     1.0000    no
NDCG@5           0.3702    0.3754 +   +0.0052   1.588     0.2531     1.0000    no
NDCG@10          0.4338    0.4385 +   +0.0047   1.528     0.2660     1.0000    no
NDCG@20          0.4770    

# Testing simulations

## XGBoost baseline vs baseline simulation 

In [1]:
import json
import numpy as np
from scipy import stats
from pathlib import Path

base = Path(r"C:\Users\cieci\OneDrive\Dokumenty\GitHub\bachelor.project\models")

xgb_baseline_path = base / "4_XGBoost_model" / "results" / "results_xgb_baseline_multiseed.json"
xgb_sim_path = base / "9_simulation_xgb" / "results" / "results_xgb_baseline_sim_multiseed.json"

with open(xgb_baseline_path, "r", encoding="utf-8") as f:
    xgb_baseline = json.load(f)

with open(xgb_sim_path, "r", encoding="utf-8") as f:
    xgb_sim = json.load(f)

METRICS = [
    "HitRate@1", "HitRate@5", "HitRate@10", "HitRate@20",
    "NDCG@1", "NDCG@5", "NDCG@10", "NDCG@20",
    "MRR",
]

ALPHA = 0.05

def extract(data, split, metric):
    key = f"per_seed_{split}"
    return np.array([seed_result[metric] for seed_result in data[key]], dtype=float)

def run_tests(original_data, simulation_data, split="test"):
    results = []

    for metric in METRICS:
        original = extract(original_data, split, metric)
        simulation = extract(simulation_data, split, metric)

        diff = simulation - original

        t_stat, p_val = stats.ttest_rel(simulation, original)

        results.append({
            "metric": metric,
            "mean_original": original.mean(),
            "mean_simulation": simulation.mean(),
            "diff": diff.mean(),
            "diff_std": diff.std(ddof=1),
            "t": t_stat,
            "p": p_val,
            "sig": p_val < ALPHA,
        })

    return results

def print_table(results, title):
    print(f"\n{'═' * 90}")
    print(f"  {title}")
    print(f"{'═' * 90}")
    print(
        f"{'Metric':<14} "
        f"{'Original':>10} "
        f"{'Simulation':>12} "
        f"{'Δ(sim-orig)':>13} "
        f"{'t':>9} "
        f"{'p':>10} "
        f"sig?"
    )
    print("─" * 90)

    for r in results:
        flag = "✓ YES" if r["sig"] else "no"
        print(
            f"{r['metric']:<14} "
            f"{r['mean_original']:>10.4f} "
            f"{r['mean_simulation']:>12.4f} "
            f"{r['diff']:>+13.4f} "
            f"{r['t']:>9.3f} "
            f"{r['p']:>10.4f} "
            f"{flag}"
        )

    print("─" * 90)
    n_sig = sum(r["sig"] for r in results)
    print(f"Significant at α={ALPHA}: {n_sig}/{len(results)} metrics")

def cohens_d_paired(original_data, simulation_data, split, metric):
    original = extract(original_data, split, metric)
    simulation = extract(simulation_data, split, metric)
    diff = simulation - original

    std = diff.std(ddof=1)
    if std == 0:
        return np.nan

    return diff.mean() / std

def interpret_d(d):
    ad = abs(d)
    if ad < 0.2:
        return "negligible"
    if ad < 0.5:
        return "small"
    if ad < 0.8:
        return "medium"
    return "large"

all_results = {}

for split in ("val", "test"):
    results = run_tests(xgb_baseline, xgb_sim, split)
    all_results[split] = results
    print_table(results, f"XGBoost Baseline vs Simulation — {split.upper()}")

print("\n\nEffect sizes for significant results:")
print(f"{'Split':<6} {'Metric':<14} {'Cohen d':>10}  interpretation")
print("─" * 50)

for split in ("val", "test"):
    for r in all_results[split]:
        if r["sig"]:
            d = cohens_d_paired(xgb_baseline, xgb_sim, split, r["metric"])
            print(f"{split:<6} {r['metric']:<14} {d:>10.3f}  {interpret_d(d)}")


══════════════════════════════════════════════════════════════════════════════════════════
  XGBoost Baseline vs Simulation — VAL
══════════════════════════════════════════════════════════════════════════════════════════
Metric           Original   Simulation   Δ(sim-orig)         t          p sig?
──────────────────────────────────────────────────────────────────────────────────────────
HitRate@1          0.7744       0.7757       +0.0013     0.390     0.7341 no
HitRate@5          0.8625       0.8635       +0.0010     0.438     0.7043 no
HitRate@10         0.9002       0.9045       +0.0043     1.613     0.2480 no
HitRate@20         0.9399       0.9405       +0.0006     0.682     0.5658 no
NDCG@1             0.7744       0.7757       +0.0013     0.390     0.7341 no
NDCG@5             0.8214       0.8222       +0.0009     0.311     0.7852 no
NDCG@10            0.8336       0.8354       +0.0019     0.720     0.5463 no
NDCG@20            0.8436       0.8446       +0.0010     0.404     0.

## XGBoost network vs network simulation 

In [14]:
import json
import numpy as np
from scipy import stats
from pathlib import Path

base = Path(r"C:\Users\cieci\OneDrive\Dokumenty\GitHub\bachelor.project\models")

xgb_baseline_path = base / "4_XGBoost_model" / "results" / "results_xgb_baseline_multiseed.json"
xgb_sim_path = base / "9_simulation_xgb" / "results" / "results_xgb_baseline_sim_multiseed.json"

with open(xgb_baseline_path, "r", encoding="utf-8") as f:
    xgb_baseline = json.load(f)

with open(xgb_sim_path, "r", encoding="utf-8") as f:
    xgb_sim = json.load(f)

METRICS = [
    "HitRate@1", "HitRate@5", "HitRate@10", "HitRate@20",
    "NDCG@1", "NDCG@5", "NDCG@10", "NDCG@20",
    "MRR",
]

ALPHA = 0.05


def extract(data, split, metric):
    key = f"per_seed_{split}"
    return np.array([seed_result[metric] for seed_result in data[key]], dtype=float)


def holm_correction(p_values, alpha=0.05):
    """
    Holm-Bonferroni correction.
    Returns adjusted p-values and significance decisions.
    """
    p_values = np.asarray(p_values, dtype=float)
    m = len(p_values)

    order = np.argsort(p_values)
    sorted_p = p_values[order]

    adjusted_sorted = np.empty(m)

    for i, p in enumerate(sorted_p):
        adjusted_sorted[i] = (m - i) * p

    adjusted_sorted = np.maximum.accumulate(adjusted_sorted)
    adjusted_sorted = np.minimum(adjusted_sorted, 1.0)

    adjusted = np.empty(m)
    adjusted[order] = adjusted_sorted

    significant = adjusted < alpha
    return adjusted, significant


def run_tests(original_data, simulation_data, split="test"):
    results = []

    for metric in METRICS:
        original = extract(original_data, split, metric)
        simulation = extract(simulation_data, split, metric)

        diff = simulation - original

        t_stat, p_val = stats.ttest_rel(simulation, original)

        results.append({
            "metric": metric,
            "mean_original": original.mean(),
            "mean_simulation": simulation.mean(),
            "diff": diff.mean(),
            "diff_std": diff.std(ddof=1),
            "t": t_stat,
            "p_raw": p_val,
        })

    raw_p_values = [r["p_raw"] for r in results]
    p_holm, sig_holm = holm_correction(raw_p_values, alpha=ALPHA)

    for r, p_adj, sig in zip(results, p_holm, sig_holm):
        r["p_holm"] = p_adj
        r["sig"] = sig

    return results


def print_table(results, title):
    print(f"\n{'═' * 105}")
    print(f"  {title}")
    print(f"{'═' * 105}")
    print(
        f"{'Metric':<14} "
        f"{'Original':>10} "
        f"{'Simulation':>12} "
        f"{'Δ(sim-orig)':>13} "
        f"{'t':>9} "
        f"{'p_raw':>10} "
        f"{'p_holm':>10} "
        f"sig?"
    )
    print("─" * 105)

    for r in results:
        flag = "✓ YES" if r["sig"] else "no"

        print(
            f"{r['metric']:<14} "
            f"{r['mean_original']:>10.4f} "
            f"{r['mean_simulation']:>12.4f} "
            f"{r['diff']:>+13.4f} "
            f"{r['t']:>9.3f} "
            f"{r['p_raw']:>10.4f} "
            f"{r['p_holm']:>10.4f} "
            f"{flag}"
        )

    print("─" * 105)
    n_sig = sum(r["sig"] for r in results)
    print(f"Significant after Holm correction at α={ALPHA}: {n_sig}/{len(results)} metrics")


def cohens_d_paired(original_data, simulation_data, split, metric):
    original = extract(original_data, split, metric)
    simulation = extract(simulation_data, split, metric)
    diff = simulation - original

    std = diff.std(ddof=1)
    if std == 0:
        return np.inf if diff.mean() != 0 else 0.0

    return diff.mean() / std


def interpret_d(d):
    ad = abs(d)
    if ad < 0.2:
        return "negligible"
    if ad < 0.5:
        return "small"
    if ad < 0.8:
        return "medium"
    return "large"


all_results = {}

for split in ("val", "test"):
    results = run_tests(xgb_baseline, xgb_sim, split)
    all_results[split] = results
    print_table(results, f"XGBoost Baseline vs Simulation — {split.upper()}")

print("\n\nEffect sizes for Holm-significant results:")
print(f"{'Split':<6} {'Metric':<14} {'Cohen d':>10}  interpretation")
print("─" * 50)

for split in ("val", "test"):
    for r in all_results[split]:
        if r["sig"]:
            d = cohens_d_paired(xgb_baseline, xgb_sim, split, r["metric"])
            print(f"{split:<6} {r['metric']:<14} {d:>10.3f}  {interpret_d(d)}")


═════════════════════════════════════════════════════════════════════════════════════════════════════════
  XGBoost Baseline vs Simulation — VAL
═════════════════════════════════════════════════════════════════════════════════════════════════════════
Metric           Original   Simulation   Δ(sim-orig)         t      p_raw     p_holm sig?
─────────────────────────────────────────────────────────────────────────────────────────────────────────
HitRate@1          0.7744       0.7757       +0.0013     0.390     0.7341     1.0000 no
HitRate@5          0.8625       0.8635       +0.0010     0.438     0.7043     1.0000 no
HitRate@10         0.9002       0.9045       +0.0043     1.613     0.2480     1.0000 no
HitRate@20         0.9399       0.9405       +0.0006     0.682     0.5658     1.0000 no
NDCG@1             0.7744       0.7757       +0.0013     0.390     0.7341     1.0000 no
NDCG@5             0.8214       0.8222       +0.0009     0.311     0.7852     1.0000 no
NDCG@10            0.833

# XGBoost experiments

In [2]:
import json
import numpy as np
from scipy import stats
from pathlib import Path

BASE_MAIN = Path(r"C:\Users\cieci\OneDrive\Dokumenty\GitHub\bachelor.project\models\4_XGBoost_model\results")
BASE_VARIANTS = Path(r"C:\Users\cieci\OneDrive\Dokumenty\GitHub\bachelor.project\models\8_xgb_variations\results\multiseed")

PATHS = {
    "baseline": BASE_MAIN / "results_xgb_baseline_multiseed.json",
    "network": BASE_MAIN / "results_xgb_network_multiseed.json",

    "baseline_game_emb_0": BASE_VARIANTS / "results_xgb_baseline+emb_0_multiseed.json",
    "embeddings_only": BASE_VARIANTS / "results_embeddings_only_multiseed.json",
    "embeddings_basic_user_info": BASE_VARIANTS / "results_xgb_embeddings+unique_genres_played+total_playtime_minutes_multiseed.json",

    "baseline_without_user_count": BASE_VARIANTS / "results_xgb_baseline-user_count_multiseed.json",
    "baseline_without_user_count_game_emb_0": BASE_VARIANTS / "results_xgb_baseline+emb_0-user_count_multiseed.json",

    "user_count_game_emb_0": BASE_VARIANTS / "results_xgb_emb0+user_count_multiseed.json",
}

METRICS = [
    "HitRate@1", "HitRate@5", "HitRate@10", "HitRate@20",
    "NDCG@1", "NDCG@5", "NDCG@10", "NDCG@20",
    "MRR",
]

ALPHA = 0.05


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


DATA = {name: load_json(path) for name, path in PATHS.items()}


COMPARISONS = [
    ("Baseline + game_emb_0 vs Baseline", "baseline", "baseline_game_emb_0"),
    ("Embeddings only vs Baseline", "baseline", "embeddings_only"),
    ("Embeddings + basic user information vs Embeddings only", "embeddings_only", "embeddings_basic_user_info"),
    ("Baseline without user_count vs Baseline", "baseline", "baseline_without_user_count"),
    ("Baseline without user_count + game_emb_0 vs Baseline without user_count", "baseline_without_user_count", "baseline_without_user_count_game_emb_0"),
    ("user_count + game_emb_0 vs Baseline", "baseline", "user_count_game_emb_0"),
    ("user_count + game_emb_0 vs Network", "network", "user_count_game_emb_0"),
]


def extract(data, split, metric):
    key = f"per_seed_{split}"
    return np.array([seed_result[metric] for seed_result in data[key]], dtype=float)


def holm_correction(p_values, alpha=0.05):
    p_values = np.asarray(p_values, dtype=float)
    m = len(p_values)

    order = np.argsort(p_values)
    sorted_p = p_values[order]

    adjusted_sorted = np.empty(m)

    for i, p in enumerate(sorted_p):
        adjusted_sorted[i] = (m - i) * p

    adjusted_sorted = np.maximum.accumulate(adjusted_sorted)
    adjusted_sorted = np.minimum(adjusted_sorted, 1.0)

    adjusted = np.empty(m)
    adjusted[order] = adjusted_sorted

    significant = adjusted < alpha
    return adjusted, significant


def cohens_d_paired(original, variant):
    diff = variant - original
    std = diff.std(ddof=1)

    if std == 0:
        return np.inf if diff.mean() != 0 else 0.0

    return diff.mean() / std


def interpret_d(d):
    ad = abs(d)
    if ad < 0.2:
        return "negligible"
    if ad < 0.5:
        return "small"
    if ad < 0.8:
        return "medium"
    return "large"


def run_tests(original_data, variant_data, split="test"):
    results = []

    for metric in METRICS:
        original = extract(original_data, split, metric)
        variant = extract(variant_data, split, metric)

        diff = variant - original

        t_stat, p_val = stats.ttest_rel(variant, original)

        d = cohens_d_paired(original, variant)

        results.append({
            "metric": metric,
            "mean_original": original.mean(),
            "mean_variant": variant.mean(),
            "diff": diff.mean(),
            "diff_std": diff.std(ddof=1),
            "t": t_stat,
            "p_raw": p_val,
            "cohens_d": d,
            "effect": interpret_d(d),
        })

    raw_p_values = [r["p_raw"] for r in results]
    p_holm, sig_holm = holm_correction(raw_p_values, alpha=ALPHA)

    for r, p_adj, sig in zip(results, p_holm, sig_holm):
        r["p_holm"] = p_adj
        r["sig"] = bool(sig)

    return results


def print_table(results, title):
    print(f"\n{'═' * 125}")
    print(f"  {title}")
    print(f"{'═' * 125}")
    print(
        f"{'Metric':<14} "
        f"{'Original':>10} "
        f"{'Variant':>10} "
        f"{'Δ(var-orig)':>13} "
        f"{'t':>9} "
        f"{'p_raw':>10} "
        f"{'p_holm':>10} "
        f"{'sig?':>8} "
        f"{'d':>9} "
        f"{'effect':>12}"
    )
    print("─" * 125)

    for r in results:
        flag = "✓ YES" if r["sig"] else "no"

        print(
            f"{r['metric']:<14} "
            f"{r['mean_original']:>10.4f} "
            f"{r['mean_variant']:>10.4f} "
            f"{r['diff']:>+13.4f} "
            f"{r['t']:>9.3f} "
            f"{r['p_raw']:>10.4g} "
            f"{r['p_holm']:>10.4g} "
            f"{flag:>8} "
            f"{r['cohens_d']:>9.3f} "
            f"{r['effect']:>12}"
        )

    print("─" * 125)
    n_sig = sum(r["sig"] for r in results)
    print(f"Significant after Holm correction at α={ALPHA}: {n_sig}/{len(results)} metrics")


all_results = {}

for split in ("val", "test"):
    for title, original_name, variant_name in COMPARISONS:
        results = run_tests(DATA[original_name], DATA[variant_name], split=split)
        all_results[(split, title)] = results

        print_table(
            results,
            f"{title} — {split.upper()}"
        )


═════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
  Baseline + game_emb_0 vs Baseline — VAL
═════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
Metric           Original    Variant   Δ(var-orig)         t      p_raw     p_holm     sig?         d       effect
─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
HitRate@1          0.7744     0.9841       +0.2097   296.932  1.134e-05  6.359e-05    ✓ YES   171.434        large
HitRate@5          0.8625     0.9915       +0.1290    90.996  0.0001207  0.0003622    ✓ YES    52.536        large
HitRate@10         0.9002     0.9944       +0.0942    64.341  0.0002415  0.0004829    ✓ YES    37.147        large
HitRate@20         0.9399     0.9963       +0.0564    53.811  0.0003452  0.0004829    ✓ YES    31.068        large
NDCG

In [3]:
import json
import numpy as np
from scipy import stats
from pathlib import Path

BASE_MAIN = Path(r"C:\Users\cieci\OneDrive\Dokumenty\GitHub\bachelor.project\models\4_XGBoost_model\results")
BASE_VARIANTS = Path(r"C:\Users\cieci\OneDrive\Dokumenty\GitHub\bachelor.project\models\8_xgb_variations\results\multiseed")

PATHS = {
    "baseline": BASE_MAIN / "results_xgb_baseline_multiseed.json",

    "baseline_log_user_count": BASE_VARIANTS / "results_xgb_baseline_log(user_count)_multiseed.json",
    "baseline_fake_emb_0": BASE_VARIANTS / "results_xgb_baseline+fake_emb_0_multiseed.json",
}

METRICS = [
    "HitRate@1", "HitRate@5", "HitRate@10", "HitRate@20",
    "NDCG@1", "NDCG@5", "NDCG@10", "NDCG@20",
    "MRR",
]

ALPHA = 0.05


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


DATA = {name: load_json(path) for name, path in PATHS.items()}


COMPARISONS = [
    ("Baseline + log(user_count) vs Baseline", "baseline", "baseline_log_user_count"),
    ("Baseline + fake emb_0 vs Baseline", "baseline", "baseline_fake_emb_0"),
]


def extract(data, split, metric):
    key = f"per_seed_{split}"
    return np.array([seed_result[metric] for seed_result in data[key]], dtype=float)


def holm_correction(p_values, alpha=0.05):
    p_values = np.asarray(p_values, dtype=float)
    m = len(p_values)

    order = np.argsort(p_values)
    sorted_p = p_values[order]

    adjusted_sorted = np.empty(m)

    for i, p in enumerate(sorted_p):
        adjusted_sorted[i] = (m - i) * p

    adjusted_sorted = np.maximum.accumulate(adjusted_sorted)
    adjusted_sorted = np.minimum(adjusted_sorted, 1.0)

    adjusted = np.empty(m)
    adjusted[order] = adjusted_sorted

    significant = adjusted < alpha
    return adjusted, significant


def cohens_d_paired(original, variant):
    diff = variant - original
    std = diff.std(ddof=1)

    if std == 0:
        return np.inf if diff.mean() != 0 else 0.0

    return diff.mean() / std


def interpret_d(d):
    ad = abs(d)
    if ad < 0.2:
        return "negligible"
    if ad < 0.5:
        return "small"
    if ad < 0.8:
        return "medium"
    return "large"


def run_tests(original_data, variant_data, split="test"):
    results = []

    for metric in METRICS:
        original = extract(original_data, split, metric)
        variant = extract(variant_data, split, metric)

        diff = variant - original

        t_stat, p_val = stats.ttest_rel(variant, original)

        d = cohens_d_paired(original, variant)

        results.append({
            "metric": metric,
            "mean_original": original.mean(),
            "mean_variant": variant.mean(),
            "diff": diff.mean(),
            "diff_std": diff.std(ddof=1),
            "t": t_stat,
            "p_raw": p_val,
            "cohens_d": d,
            "effect": interpret_d(d),
        })

    raw_p_values = [r["p_raw"] for r in results]
    p_holm, sig_holm = holm_correction(raw_p_values, alpha=ALPHA)

    for r, p_adj, sig in zip(results, p_holm, sig_holm):
        r["p_holm"] = p_adj
        r["sig"] = bool(sig)

    return results


def print_table(results, title):
    print(f"\n{'═' * 125}")
    print(f"  {title}")
    print(f"{'═' * 125}")
    print(
        f"{'Metric':<14} "
        f"{'Baseline':>10} "
        f"{'Variant':>10} "
        f"{'Δ(var-base)':>13} "
        f"{'t':>9} "
        f"{'p_raw':>10} "
        f"{'p_holm':>10} "
        f"{'sig?':>8} "
        f"{'d':>9} "
        f"{'effect':>12}"
    )
    print("─" * 125)

    for r in results:
        flag = "✓ YES" if r["sig"] else "no"

        print(
            f"{r['metric']:<14} "
            f"{r['mean_original']:>10.4f} "
            f"{r['mean_variant']:>10.4f} "
            f"{r['diff']:>+13.4f} "
            f"{r['t']:>9.3f} "
            f"{r['p_raw']:>10.4g} "
            f"{r['p_holm']:>10.4g} "
            f"{flag:>8} "
            f"{r['cohens_d']:>9.3f} "
            f"{r['effect']:>12}"
        )

    print("─" * 125)
    n_sig = sum(r["sig"] for r in results)
    print(f"Significant after Holm correction at α={ALPHA}: {n_sig}/{len(results)} metrics")


all_results = {}

for split in ("val", "test"):
    for title, original_name, variant_name in COMPARISONS:
        results = run_tests(DATA[original_name], DATA[variant_name], split=split)
        all_results[(split, title)] = results

        print_table(
            results,
            f"{title} — {split.upper()}"
        )


═════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
  Baseline + log(user_count) vs Baseline — VAL
═════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
Metric           Baseline    Variant   Δ(var-base)         t      p_raw     p_holm     sig?         d       effect
─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
HitRate@1          0.7744     0.7708       -0.0037    -1.421     0.2911          1       no    -0.821        large
HitRate@5          0.8625     0.8576       -0.0049    -1.245     0.3391          1       no    -0.719       medium
HitRate@10         0.9002     0.8980       -0.0022    -0.975     0.4324          1       no    -0.563       medium
HitRate@20         0.9399     0.9386       -0.0013    -0.735     0.5386          1       no    -0.425        small